In [97]:
import pandas as pd
import numpy as np
import gpxpy
from geopy.distance import geodesic
from pathlib import Path
from tqdm import tqdm
import gzip
import tempfile
from fitparse import FitFile
import os
import pickle
from IPython.display import display
import folium
import plotly.express as px

In [98]:
with open('data/dataset_points.pkl', 'rb') as f:
    dataset_points = pickle.load(f)
print('Type:', type(dataset_points))

try:
    display(dataset_points.head())
except Exception:
    import pandas as pd
    try:
        df = pd.DataFrame(dataset_points)
        display(df.head())
    except Exception as e:
        print('Impossible de convertir en DataFrame:', e)
        print('Aperçu (repr):', repr(dataset_points)[:500])

Type: <class 'dict'>


,10035191179.gpx,10044645169.gpx,10082933077.gpx,10120058183.gpx,10123997315.gpx,10143116542.gpx,10163801331.gpx,10204241096.gpx,10244651939.gpx,10268040578.gpx,...,9858926816.gpx,9893728904.gpx,9897561309.gpx,9899495874.gpx,9903587740.gpx,9906191594.gpx,9909769363.gpx,9937200447.gpx,9980387967.gpx,9999027530.gpx
points,"[(43.318708, 1.94741), (43.318697, 1.947409), ...","[(43.490345, 2.374815), (43.490344, 2.374824),...","[(43.522684, 1.550141), (43.522683, 1.550141),...","[(43.522646, 1.550131), (43.522649, 1.550125),...","[(43.522717, 1.550294), (43.522717, 1.550294),...","[(43.318757, 1.947415), (43.318749, 1.947413),...","[(43.522692, 1.550168), (43.522691, 1.550164),...","[(43.522674, 1.550104), (43.522686, 1.550099),...","[(43.52267, 1.549985), (43.522671, 1.549979), ...","[(43.522656, 1.550096), (43.522657, 1.550093),...",...,"[(43.522639, 1.550189), (43.52264, 1.550194), ...","[(42.819712, 0.403564), (42.819718, 0.403571),...","[(42.81968, 0.403379), (42.819679, 0.403377), ...","[(42.81589, 0.430224), (42.81589, 0.430224), (...","[(42.819708, 0.403451), (42.819719, 0.403458),...","[(42.827819, 0.169555), (42.82782, 0.169555), ...","[(42.819709, 0.403498), (42.819708, 0.403497),...","[(43.522659, 1.550114), (43.52266, 1.550116), ...","[(43.522696, 1.550106), (43.522703, 1.55011), ...","[(43.522647, 1.55006), (43.522655, 1.55008), (..."
elevations,"[178.2, 178.1, 178.1, 178.1, 178.1, 178.0, 178...","[255.4, 255.4, 255.4, 255.4, 255.4, 255.4, 255...","[177.4, 177.4, 177.4, 177.5, 177.6, 177.7, 177...","[177.0, 177.0, 177.1, 177.1, 177.2, 177.2, 177...","[177.8, 177.8, 177.8, 177.8, 177.9, 178.0, 178...","[178.3, 178.3, 178.3, 178.2, 178.2, 178.2, 178...","[177.5, 177.5, 177.6, 177.7, 177.8, 177.9, 178...","[177.3, 177.4, 177.6, 177.7, 177.9, 178.0, 178...","[177.2, 177.3, 177.3, 177.4, 177.4, 177.6, 177...","[177.1, 177.1, 177.1, 177.1, 177.2, 177.2, 177...",...,"[176.7, 176.7, 176.9, 177.0, 177.0, 177.1, 177...","[956.8, 956.8, 956.8, 956.8, 956.8, 956.8, 956...","[956.6, 956.6, 956.6, 956.6, 956.6, 956.6, 956...","[1292.4, 1292.4, 1292.4, 1292.4, 1292.4, 1292....","[957.0, 957.0, 957.0, 957.0, 957.0, 957.0, 956...","[1863.9, 1863.9, 1864.0, 1864.1, 1864.2, 1864....","[957.0, 957.0, 957.0, 956.9, 956.9, 956.8, 956...","[176.9, 176.9, 177.0, 177.1, 177.1, 177.1, 177...","[177.4, 177.5, 177.6, 177.8, 178.1, 178.3, 178...","[176.7, 176.9, 177.1, 177.3, 177.4, 177.5, 178..."
distance_km,42.75,83.01,80.83,33.66,76.51,72.31,88.61,60.16,84.57,31.09,...,46.13,20.63,72.86,6.04,21.14,10.08,62.29,35.57,66.27,92.22
denivele_m,527.4,1641.0,896.4,381.6,995.5,1145.4,1112.7,896.9,1092.3,412.9,...,651.0,414.0,1783.2,252.1,687.9,368.3,1869.4,436.3,1001.1,1172.6


In [99]:
random_gpx = np.random.choice(df.columns)
random_gpx

'9193431472.gpx'

In [100]:
# Identifiant du trajet (inclure l'extension si présente dans les colonnes)
trip_key = random_gpx

In [101]:
# Récupérer la ligne/les points pour la clé fournie
def get_points_for_key(ds, key):
    row = None
    if isinstance(ds, dict):
        row = ds.get(key)
    elif isinstance(ds, (pd.DataFrame, pd.Series)):
        try:
            if key in ds.index:
                row = ds.loc[key]
            else:
                # chercher dans colonnes communes
                candidates = ['id','activity_id','activityId','name']
                found = None
                for c in candidates:
                    if c in ds.columns:
                        sel = ds[ds[c]==key]
                        if not sel.empty:
                            found = sel.iloc[0]
                            break
                row = found
        except Exception:
            # protection si ds.index membership échoue
            row = None
    else:
        row = None
    return row

In [102]:
# Normaliser la liste de points vers (lat, lon) floats
def normalize_points(points):
    pts = []
    if points is None:
        return pts
    import pandas as _pd
    # si c'est un DataFrame/Series de points avec colonnes lat/lon
    try:
        cols = set(points.columns) if hasattr(points, 'columns') else set()
    except Exception:
        cols = set()
    if isinstance(points, (_pd.DataFrame, _pd.Series)) and {'lat','lon'}.issubset(cols):
        for _, r in points.iterrows():
            pts.append((float(r['lat']), float(r['lon'])))
        return pts
    # sinon itérer sur les éléments
    for p in points:
        if p is None:
            continue
        # tuple/list [lat, lon]
        if isinstance(p, (list, tuple)) and len(p) >= 2:
            try:
                lat = float(p[0]); lon = float(p[1])
                pts.append((lat, lon))
                continue
            except Exception:
                pass
        # dict with keys
        if isinstance(p, dict):
            lat = p.get('lat') or p.get('latitude') or p.get('Latitude') or p.get('LAT') or p.get('lat_deg')
            lon = p.get('lon') or p.get('lng') or p.get('longitude') or p.get('LONG') or p.get('lon_deg')
            if lat is not None and lon is not None:
                try:
                    pts.append((float(lat), float(lon)))
                    continue
                except Exception:
                    pass
        # object with attributes
        if hasattr(p, 'latitude') and hasattr(p, 'longitude'):
            try:
                pts.append((float(getattr(p,'latitude')), float(getattr(p,'longitude'))))
                continue
            except Exception:
                pass
    return pts

In [103]:
coordinates = df.iloc[0]
elevation = df.iloc[1]
distance = df.iloc[2]
denivele = df.iloc[3]

In [104]:

# Récupérer la ligne et les points
row = get_points_for_key(dataset_points, trip_key)
if row is None:
    raise ValueError(f"Trajet '{trip_key}' introuvable dans dataset_points")

# Extraire la colonne/clef 'points' si présente
pts_raw = None
if isinstance(row, dict):
    pts_raw = row.get('points') or row.get('Points') or row.get('track') or row.get('points_list')
elif isinstance(row, pd.Series):
    for c in ['points','Points','track','points_list']:
        if c in row.index:
            pts_raw = row[c]
            break
elif isinstance(row, (list, tuple)):
    pts_raw = row
else:
    try:
        pts_raw = getattr(row, 'points')
    except Exception:
        pts_raw = None
if pts_raw is None:
    raise ValueError('Impossible de trouver la colonne/clé "points" pour ce trajet')
pts = normalize_points(pts_raw)
if len(pts) == 0:
    raise ValueError('Aucun point GPS valide trouvé pour ce trajet')

# Centrer la carte sur la moyenne des coordonnées
lats = [p[0] for p in pts]; lons = [p[1] for p in pts]
center = (float(np.mean(lats)), float(np.mean(lons)))
m = folium.Map(location=center, zoom_start=13)
# Ajout de la polyline reliant les points
folium.PolyLine(pts, color='blue', weight=4, opacity=0.7).add_to(m)
# Marqueur début / fin
folium.Marker(pts[0], popup='Start', icon=folium.Icon(color='green')).add_to(m)
folium.Marker(pts[-1], popup='End', icon=folium.Icon(color='red')).add_to(m)
# Afficher la carte dans le notebook
print(f"Distance : {distance[trip_key]} km Denivele : {denivele[trip_key]} m")
display(m)


Distance : 70.78 km Denivele : 1153.3 m


In [105]:
fig = px.line(x=range(len(elevation[trip_key])), y=elevation[trip_key], title='Profil du parcours', labels={"x": "Distance", "y": "Altitude (m)"})
fig.update_traces(fill='tozeroy', fillcolor='rgba(255, 0, 0, 0.3)', line=dict(color='orange', width=2))
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(range=[0.95*min(elevation[trip_key]), 1.05*max(elevation[trip_key])], tick0=0)
fig